# Study 1 — 01a: Corpus Quality Assessment

Assesses label completeness, flag completeness, confidence distribution, and structural coherence of the Phase 3 auto-coded corpus (320 traces, 73,383 sentences).

**Outputs:**
- `outputs/study1_analysis/tables/quality_assessment.csv`
- `outputs/study1_analysis/figures/confidence_by_label.png`

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
from study1_helpers import *

traces = load_traces()
df = build_sentence_df(traces)

print(f'Corpus: {df.trace_key.nunique()} traces, {len(df):,} sentences')
print(f'Sets: {df.set.value_counts().to_dict()}')
print(f'Tasks: {df.task_id.value_counts().sort_index().to_dict()}')
print(f'Completed: {df.groupby("trace_key").completed.first().sum()} / {df.trace_key.nunique()}')
print(f'Sentences per trace: mean={df.groupby("trace_key").size().mean():.1f}, '
      f'median={df.groupby("trace_key").size().median():.0f}, '
      f'SD={df.groupby("trace_key").size().std():.1f}')

# Traces by set and task
display(df.groupby(['set', 'task_id']).trace_key.nunique().unstack().fillna(0).astype(int))

## 1a. Label Completeness

In [ ]:
section_header('1a. Label Completeness')

qa_rows = []

# Missing or invalid micro_labels
missing_micro = df['micro_label'].isna().sum()
invalid_micro = (~df['micro_label'].isin(MICRO_LABELS) & df['micro_label'].notna()).sum()
invalid_micro_vals = df.loc[~df['micro_label'].isin(MICRO_LABELS) & df['micro_label'].notna(), 'micro_label'].unique()
pct_valid_micro = (len(df) - missing_micro - invalid_micro) / len(df) * 100

print(f'Micro-label coverage: {pct_valid_micro:.2f}%')
print(f'  Missing: {missing_micro}')
print(f'  Invalid: {invalid_micro}  {list(invalid_micro_vals) if invalid_micro > 0 else ""}')

qa_rows.append({'check': 'micro_label_valid_pct', 'value': f'{pct_valid_micro:.2f}%', 'n_issues': missing_micro + invalid_micro})

# Missing macro_labels
missing_macro = df['macro_label'].isna().sum()
print(f'\nMacro-label coverage: {(len(df) - missing_macro) / len(df) * 100:.2f}%')
print(f'  Missing: {missing_macro}')

# Verify macro = MACRO_MAP[micro]
expected_macro = df['micro_label'].map(MACRO_MAP)
mismatch = (df['macro_label'] != expected_macro) & df['micro_label'].notna() & df['macro_label'].notna()
n_mismatch = mismatch.sum()
print(f'\nMacro/micro mismatches: {n_mismatch}')
if n_mismatch > 0:
    print(df.loc[mismatch, ['trace_key', 'sentence_id', 'micro_label', 'macro_label']].head(10))

qa_rows.append({'check': 'macro_label_valid_pct', 'value': f'{(len(df) - missing_macro) / len(df) * 100:.2f}%', 'n_issues': missing_macro})
qa_rows.append({'check': 'macro_micro_mismatch', 'value': str(n_mismatch), 'n_issues': n_mismatch})

## 1b. Flag Completeness

In [ ]:
section_header('1b. Flag Completeness')

# TEST sentences: test_context and specificity
test_df = df[df['micro_label'] == 'TEST']
n_test = len(test_df)

tc_valid = test_df['test_context'].isin(VALID_TEST_CONTEXT).sum()
tc_missing = test_df['test_context'].isna().sum()
tc_invalid = n_test - tc_valid - tc_missing
print(f'TEST sentences: {n_test}')
print(f'  test_context valid: {tc_valid} ({tc_valid/n_test*100:.1f}%)')
print(f'  test_context missing: {tc_missing}')
print(f'  test_context invalid: {tc_invalid}')

sp_valid = test_df['specificity'].isin(VALID_SPECIFICITY).sum()
sp_missing = test_df['specificity'].isna().sum()
sp_invalid = n_test - sp_valid - sp_missing
print(f'  specificity valid: {sp_valid} ({sp_valid/n_test*100:.1f}%)')
print(f'  specificity missing: {sp_missing}')
print(f'  specificity invalid: {sp_invalid}')

qa_rows.append({'check': 'test_context_valid_pct', 'value': f'{tc_valid/n_test*100:.1f}%', 'n_issues': tc_missing + tc_invalid})
qa_rows.append({'check': 'specificity_valid_pct', 'value': f'{sp_valid/n_test*100:.1f}%', 'n_issues': sp_missing + sp_invalid})

# JUDGE sentences: judgement
judge_df = df[df['micro_label'] == 'JUDGE']
n_judge = len(judge_df)
j_valid = judge_df['judgement'].isin(VALID_JUDGEMENT).sum()
j_missing = judge_df['judgement'].isna().sum()
j_invalid = n_judge - j_valid - j_missing
print(f'\nJUDGE sentences: {n_judge}')
print(f'  judgement valid: {j_valid} ({j_valid/n_judge*100:.1f}%)')
print(f'  judgement missing: {j_missing}')
print(f'  judgement invalid: {j_invalid}')

qa_rows.append({'check': 'judgement_valid_pct', 'value': f'{j_valid/n_judge*100:.1f}%', 'n_issues': j_missing + j_invalid})

# All sentences: confidence
c_valid = df['confidence'].isin(VALID_CONFIDENCE).sum()
c_missing = df['confidence'].isna().sum()
c_invalid = len(df) - c_valid - c_missing
print(f'\nAll sentences: {len(df)}')
print(f'  confidence valid: {c_valid} ({c_valid/len(df)*100:.1f}%)')
print(f'  confidence missing: {c_missing}')
print(f'  confidence invalid: {c_invalid}')

qa_rows.append({'check': 'confidence_valid_pct', 'value': f'{c_valid/len(df)*100:.1f}%', 'n_issues': c_missing + c_invalid})

# Non-TEST with test_context or specificity (should be 0)
non_test = df[df['micro_label'] != 'TEST']
non_test_tc = non_test['test_context'].notna().sum()
non_test_sp = non_test['specificity'].notna().sum()
print(f'\nNon-TEST with test_context: {non_test_tc}')
print(f'Non-TEST with specificity: {non_test_sp}')

# Non-JUDGE with judgement (should be 0)
non_judge = df[df['micro_label'] != 'JUDGE']
non_judge_j = non_judge['judgement'].notna().sum()
print(f'Non-JUDGE with judgement: {non_judge_j}')

qa_rows.append({'check': 'non_test_has_test_flags', 'value': str(non_test_tc + non_test_sp), 'n_issues': non_test_tc + non_test_sp})
qa_rows.append({'check': 'non_judge_has_judgement', 'value': str(non_judge_j), 'n_issues': non_judge_j})

## 1c. Confidence Distribution

In [ ]:
section_header('1c. Confidence Distribution')

# Overall
conf_counts = df['confidence'].value_counts()
print('Overall confidence:')
for k, v in conf_counts.items():
    print(f'  {k}: {v} ({v/len(df)*100:.1f}%)')

# Per micro-label
conf_by_micro = df.groupby('micro_label')['confidence'].value_counts().unstack(fill_value=0)
conf_by_micro['medium_pct'] = conf_by_micro.get('medium', 0) / conf_by_micro.sum(axis=1) * 100
conf_by_micro = conf_by_micro.reindex(MICRO_LABELS)
print('\nMedium-confidence % by micro-label:')
display(conf_by_micro)

# Per task
conf_by_task = df.groupby('task_id')['confidence'].value_counts().unstack(fill_value=0)
conf_by_task['medium_pct'] = conf_by_task.get('medium', 0) / conf_by_task.sum(axis=1) * 100
print('\nMedium-confidence % by task:')
display(conf_by_task)

# Visualise: stacked bar chart
fig, ax = plt.subplots(figsize=(10, 5))
conf_plot = conf_by_micro[['high', 'medium']].reindex(MICRO_LABELS)
conf_plot = conf_plot.div(conf_plot.sum(axis=1), axis=0) * 100
conf_plot.plot(kind='bar', stacked=True, ax=ax, color=['#4C72B0', '#DD8452'], edgecolor='white')
ax.set_ylabel('Percentage')
ax.set_xlabel('Micro-label')
ax.set_title('Confidence Distribution by Micro-label')
ax.legend(title='Confidence', loc='upper right')
ax.set_xticklabels(MICRO_LABELS, rotation=45, ha='right')
plt.tight_layout()
save_fig(fig, 'confidence_by_label.png')
plt.show()

## 1d. Structural Coherence Checks

In [ ]:
section_header('1d. Structural Coherence Checks')

# ORIENT in first 5 sentences?
orient_early = df[df['position_idx'] < 5].groupby('trace_key')['micro_label'].apply(
    lambda x: 'ORIENT' in x.values
)
pct_orient = orient_early.mean() * 100
print(f'Traces with ORIENT in first 5 sentences: {orient_early.sum()}/{len(orient_early)} ({pct_orient:.1f}%)')

qa_rows.append({'check': 'orient_in_first5_pct', 'value': f'{pct_orient:.1f}%', 'n_issues': int((~orient_early).sum())})

# RULE in completed traces?
completed_traces = df.groupby('trace_key').filter(lambda g: g['completed'].iloc[0])
rule_in_completed = completed_traces.groupby('trace_key')['micro_label'].apply(
    lambda x: 'RULE' in x.values
)
pct_rule = rule_in_completed.mean() * 100
n_completed = rule_in_completed.shape[0]
print(f'Completed traces with RULE: {rule_in_completed.sum()}/{n_completed} ({pct_rule:.1f}%)')
if (~rule_in_completed).any():
    missing_rule = rule_in_completed[~rule_in_completed].index.tolist()
    print(f'  Missing RULE in: {missing_rule[:10]}')

qa_rows.append({'check': 'rule_in_completed_pct', 'value': f'{pct_rule:.1f}%', 'n_issues': int((~rule_in_completed).sum())})

# Runs of 10+ identical consecutive labels
print('\nTraces with runs of 10+ identical labels:')
run_issues = []
for key, grp in df.groupby('trace_key'):
    labels = grp['micro_label'].values
    max_run = 1
    cur_run = 1
    cur_label = labels[0]
    for lbl in labels[1:]:
        if lbl == cur_label:
            cur_run += 1
            max_run = max(max_run, cur_run)
        else:
            cur_run = 1
            cur_label = lbl
    if max_run >= 10:
        run_issues.append((key, max_run, cur_label))
        print(f'  {key}: max run = {max_run}')
if not run_issues:
    print('  None found.')

qa_rows.append({'check': 'traces_with_10plus_run', 'value': str(len(run_issues)), 'n_issues': len(run_issues)})

# >70% single label dominance
print('\nTraces with >70% of a single label:')
dom_issues = []
for key, grp in df.groupby('trace_key'):
    counts = grp['micro_label'].value_counts(normalize=True)
    if counts.iloc[0] > 0.70:
        dom_issues.append((key, counts.index[0], counts.iloc[0]))
        print(f'  {key}: {counts.index[0]} = {counts.iloc[0]*100:.1f}%')
if not dom_issues:
    print('  None found.')

qa_rows.append({'check': 'traces_with_70pct_dominance', 'value': str(len(dom_issues)), 'n_issues': len(dom_issues)})

# Save quality assessment
qa_df = pd.DataFrame(qa_rows)
save_table(qa_df, 'quality_assessment.csv')
display(qa_df)